# 핸드오프 스크립트를 Colab에서 실행

`apply_kirc_to_hetionet_subgraph.py` 는 노트북이 아니라 일반 파이썬 파일이라 Colab에서 더블클릭으로 열 수 없습니다.
이 노트북이 대신 실행해 줍니다.

**필요한 것: 첫 번째 zip 하나** (`TalkFile_kirc_subgraph_*.zip`). Drive는 필요 없습니다.

1. 맨 위부터 순서대로 실행합니다.
2. 두 번째 셀에서 업로드 창이 뜨면 zip을 선택합니다.
3. 마지막 셀이 결과를 `output.zip`으로 묶어 내려받게 합니다.

압축을 푼 폴더가 이미 Drive에 있다면 첫 셀의 `HANDOFF_DIR`에 경로를 적으세요.

In [ ]:
import os, sys, shutil, subprocess, zipfile, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# 방법 A (기본): 비워 두면 zip을 업로드합니다.
# 방법 B: 압축을 푼 폴더가 Drive에 있으면 그 경로를 적습니다. 예)
#   "/content/drive/MyDrive/BML/KIRC_Hetionet_Project/2026-10-05_handoff_and_seed_analysis/team_handoff_kirc_subgraph"
HANDOFF_DIR = ""

REQUIRED = ["apply_kirc_to_hetionet_subgraph.py", "kirc_hetionet_gene_nodes.tsv",
            "subgraph_nodes.tsv", "subgraph_edges.tsv"]
SCRIPT = "apply_kirc_to_hetionet_subgraph.py"

In [ ]:
def find_handoff_zip():
    """현재 폴더의 zip 중 실행 스크립트가 들어 있는 것을 찾는다."""
    for z in sorted(Path(".").glob("*.zip")):
        try:
            with zipfile.ZipFile(z) as f:
                if any(n.endswith("/" + SCRIPT) or n == SCRIPT for n in f.namelist()):
                    return z
        except zipfile.BadZipFile:
            continue
    return None


if HANDOFF_DIR:
    if IN_COLAB and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    work = Path(HANDOFF_DIR)
else:
    zf = find_handoff_zip()
    if zf is None and IN_COLAB:
        from google.colab import files
        files.upload()                     # 업로드 창이 열립니다
        zf = find_handoff_zip()
    if zf is None:
        raise FileNotFoundError("실행 스크립트가 든 zip을 찾지 못했습니다. "
                                "TalkFile_kirc_subgraph_*.zip 을 이 폴더에 두거나 업로드하세요.")
    print("사용할 zip:", zf)
    with zipfile.ZipFile(zf) as f:
        f.extractall("handoff")
    # zip 안에는 폴더가 한 겹 있고, macOS 부산물(__MACOSX)이 섞여 있다
    found = [p.parent for p in Path("handoff").rglob(SCRIPT) if "__MACOSX" not in p.parts]
    if not found:
        raise FileNotFoundError(f"압축을 풀었지만 {SCRIPT} 를 찾지 못했습니다.")
    work = found[0]

missing = [f for f in REQUIRED if not (work / f).exists()]
if missing:
    raise FileNotFoundError(f"{work} 에 없는 파일: {missing}")

print("작업 폴더:", work.resolve())
for f in REQUIRED:
    print(f"  {f:<40s} {(work / f).stat().st_size / 1e6:>8.2f} MB")

## 실행

스크립트는 자기와 같은 폴더의 TSV 3개를 읽어서 같은 폴더 아래 `output/` 에 결과를 씁니다.
pandas가 필요하며 Colab에는 이미 설치돼 있습니다. 약 20~30초 걸립니다.

In [ ]:
r = subprocess.run([sys.executable, SCRIPT], cwd=work, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("스크립트 실행 실패")
summary = json.loads((work / "output" / "summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in summary.items() if k != "edge_counts"}, ensure_ascii=False, indent=2))

## 기대값과 대조

핸드오프 README에 적힌 기대 결과입니다. **구버전 gene 파일(19,416개)을 쓸 때의 값**이라,
다른 gene 파일로 돌리면 아래 몇 항목이 달라지는 것이 정상입니다
(최종본 19,425개로 돌리면 gene 수 3개 항목과 전체 노드·엣지가 1씩 늘어납니다).

In [ ]:
EXPECTED = {
    "kirc_gene_nodes_requested": 19289,
    "kirc_gene_nodes_in_base_subgraph": 18080,
    "pathways": 1822,
    "genes_connected_to_pathway": 8928,
    "pathway_gene_edges": 84137,
    "kidney_cancer_seed_genes": 698,
    "pathways_with_kidney_cancer_seed": 1230,
    "total_nodes": 31417,
    "total_edges": 1082475,
}
bad = 0
for k, want in EXPECTED.items():
    got = summary[k]
    ok = got == want
    bad += (not ok)
    print(f"[{'OK  ' if ok else 'DIFF'}] {k:<36s} 기대 {want:>10,}  실제 {got:>10,}")
print("\n" + ("README 기대값과 전부 일치합니다." if bad == 0 else f"{bad}개 항목이 다릅니다."))

## 결과 받기

`output/` 의 5개 파일(`nodes.tsv`, `edges.tsv`, `pathway_summary.tsv`, `gene_summary.tsv`, `summary.json`)을
`output.zip` 으로 묶습니다. Colab에서는 내려받기가 시작됩니다.

In [ ]:
out = work / "output"
for p in sorted(out.iterdir()):
    print(f"  {p.name:<22s} {p.stat().st_size / 1e6:>8.2f} MB")

archive = shutil.make_archive("output", "zip", out)
print("\n묶음:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)